In [1]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [5]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]

        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        image = preprocess_t2f(image)

        image = torch.from_numpy(image).unsqueeze(0)

        return {
            "image": image,
            "subject": subject
        }

In [6]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [8]:
timesteps = 1000

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, timesteps)

alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

In [9]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2

        embeddings = math.log(10000) / (half_dim - 1)

        embeddings = torch.exp(
            torch.arange(half_dim, device=device) * -embeddings
        )

        embeddings = t[:, None] * embeddings[None, :]

        embeddings = torch.cat(
            (embeddings.sin(), embeddings.cos()),
            dim=1
        )

        return embeddings

In [10]:
class ResBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.conv2 = nn.Conv3d(
            out_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.time_mlp = nn.Linear(
            time_dim,
            out_channels
        )

        if in_channels != out_channels:
            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        time_emb = time_emb[:, :, None, None, None]

        h = h + time_emb

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.residual(x)

In [11]:
class DownBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.resblock = ResBlock3D(
            in_channels,
            out_channels,
            time_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(self, x, t):
        h = self.resblock(x, t)

        down = self.downsample(h)

        return h, down


class UpBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock = ResBlock3D(
            out_channels + skip_channels,
            out_channels,
            time_dim
        )

    def forward(self, x, skip, t):
        x = self.upsample(x)

        x = torch.cat([x, skip], dim=1)

        x = self.resblock(x, t)

        return x

In [12]:
class UNet3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=16,
        time_dim=128
    ):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.input_conv = nn.Conv3d(
            in_channels,
            base_channels,
            kernel_size=3,
            padding=1
        )

        self.down1 = DownBlock3D(
            base_channels,
            base_channels * 2,
            time_dim
        )

        self.down2 = DownBlock3D(
            base_channels * 2,
            base_channels * 4,
            time_dim
        )

        self.down3 = DownBlock3D(
            base_channels * 4,
            base_channels * 8,
            time_dim
        )

        self.mid = ResBlock3D(
            base_channels * 8,
            base_channels * 8,
            time_dim
        )

        self.up3 = UpBlock3D(
            in_channels=base_channels * 8,
            skip_channels=base_channels * 8,
            out_channels=base_channels * 4,
            time_dim=time_dim
        )

        self.up2 = UpBlock3D(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 4,
            out_channels=base_channels * 2,
            time_dim=time_dim
        )

        self.up1 = UpBlock3D(
            in_channels=base_channels * 2,
            skip_channels=base_channels * 2,
            out_channels=base_channels,
            time_dim=time_dim
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x, t):
        t = self.time_embedding(t)

        x = self.input_conv(x)

        skip1, x = self.down1(x, t)
        skip2, x = self.down2(x, t)
        skip3, x = self.down3(x, t)

        x = self.mid(x, t)

        x = self.up3(x, skip3, t)
        x = self.up2(x, skip2, t)
        x = self.up1(x, skip1, t)

        x = self.output_conv(x)

        return x

In [13]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    device = x0.device

    sqrt_alpha_cumprod_device = sqrt_alphas_cumprod.to(device)
    sqrt_one_minus_alpha_cumprod_device = (
        sqrt_one_minus_alphas_cumprod.to(device)
    )

    sqrt_alpha_hat = (
        sqrt_alpha_cumprod_device[t]
        .view(-1, 1, 1, 1, 1)
    )

    sqrt_one_minus_alpha_hat = (
        sqrt_one_minus_alpha_cumprod_device[t]
        .view(-1, 1, 1, 1, 1)
    )

    xt = (
        sqrt_alpha_hat * x0
        + sqrt_one_minus_alpha_hat * noise
    )

    return xt, noise

In [14]:
def train_ddpm(
    model,
    train_loader,
    epochs,
    optimizer,
    device,
    start_epoch=0,
    checkpoint_dir="checkpoints"
):
    import os

    os.makedirs(checkpoint_dir, exist_ok=True)

    model.train()

    # Load previous loss history if it exists
    loss_history_path = os.path.join(
        checkpoint_dir,
        "ddpm_loss_history.npy"
    )

    if os.path.exists(loss_history_path):
        loss_history = np.load(
            loss_history_path
        ).tolist()
    else:
        loss_history = []
        fg_history = []
        bg_history = []

    # Move diffusion schedule to the same device once
    sqrt_alpha_cumprod_device = sqrt_alphas_cumprod.to(device)

    sqrt_one_minus_alpha_cumprod_device = (
        sqrt_one_minus_alphas_cumprod.to(device)
    )

    for epoch in range(start_epoch, epochs):
        epoch_loss = 0.0
        epoch_fg_loss = 0.0
        epoch_bg_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            x0 = batch["image"].to(device)

            t = torch.randint(
                0,
                timesteps,
                (x0.shape[0],),
                device=device
            )

            noise = torch.randn_like(x0)

            sqrt_alpha_hat = (
                sqrt_alpha_cumprod_device[t]
                .view(-1, 1, 1, 1, 1)
            )

            sqrt_one_minus_alpha_hat = (
                sqrt_one_minus_alpha_cumprod_device[t]
                .view(-1, 1, 1, 1, 1)
            )

            xt = (
                sqrt_alpha_hat * x0
                + sqrt_one_minus_alpha_hat * noise
            )

            predicted_noise = model(
                xt,
                t
            )

            foreground = x0 > 0
            background = ~foreground

            fg_loss = F.mse_loss(
                predicted_noise[foreground],
                noise[foreground]
            )

            bg_loss = F.mse_loss(
                predicted_noise[background],
                noise[background]
            )

            loss = (
                0.8 * fg_loss
                + 0.2 * bg_loss
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_fg_loss += fg_loss.item()
            epoch_bg_loss += bg_loss.item()

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Loss: {loss.item():.6f} | "
                    f"FG: {fg_loss.item():.6f} | "
                    f"BG: {bg_loss.item():.6f}"
                )

        avg_loss = epoch_loss / len(train_loader)
        avg_fg_loss = (
            epoch_fg_loss / len(train_loader)
        )

        avg_bg_loss = (
            epoch_bg_loss / len(train_loader)
        )
        

        loss_history.append(avg_loss)
        fg_history.append(avg_fg_loss)
        bg_history.append(avg_bg_loss)

        print(
            f"Epoch {epoch + 1} completed | "
            f"Loss: {avg_loss:.6f} | "
            f"FG: {avg_fg_loss:.6f} | "
            f"BG: {avg_bg_loss:.6f}"
        )

        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"ddpm_epoch_{epoch + 1:03d}.pt"
        )

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            path=checkpoint_path
        )

        print("Saved:", checkpoint_path)

        np.save(
            loss_history_path,
            np.array(loss_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "ddpm_fg_loss_history.npy"
            ),
            np.array(fg_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "ddpm_bg_loss_history.npy"
            ),
            np.array(bg_history)
        )

In [15]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict()
    }, path)


def load_checkpoint(model, optimizer, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return checkpoint["epoch"]

In [16]:
@torch.no_grad()
def sample_ddpm(model, shape, device, sample_timesteps=None):
    model.eval()

    if sample_timesteps is None:
        sample_timesteps = timesteps

    x = torch.randn(shape, device=device)

    for t in reversed(range(sample_timesteps)):
        t_batch = torch.full(
            (shape[0],),
            t,
            device=device,
            dtype=torch.long
        )

        beta_t = betas[t].to(device)
        alpha_t = alphas[t].to(device)
        alpha_hat_t = alphas_cumprod[t].to(device)

        predicted_noise = model(x, t_batch)

        model_mean = (
            1 / torch.sqrt(alpha_t)
        ) * (
            x
            - (
                beta_t
                / torch.sqrt(1 - alpha_hat_t)
            ) * predicted_noise
        )

        if t > 0:
            alpha_hat_prev = alphas_cumprod[t - 1].to(device)

            posterior_variance_t = (
                beta_t
                * (1 - alpha_hat_prev)
                / (1 - alpha_hat_t)
            )

            noise = torch.randn_like(x)

            x = (
                model_mean
                + torch.sqrt(posterior_variance_t) * noise
            )
        else:
            x = model_mean

    return x

In [17]:
device = torch.device("cuda")

model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

train_ddpm(
    model=model,
    train_loader=train_loader,
    epochs=10,
    optimizer=optimizer,
    device=device,
    start_epoch=0,
    checkpoint_dir="ddpm_v3_checkpoints"
)

Epoch 1/10 | Batch 10/1000 | Loss: 0.629506 | FG: 0.628884 | BG: 0.631994


Epoch 1/10 | Batch 20/1000 | Loss: 0.582243 | FG: 0.570692 | BG: 0.628444


Epoch 1/10 | Batch 30/1000 | Loss: 0.239220 | FG: 0.234439 | BG: 0.258345


Epoch 1/10 | Batch 40/1000 | Loss: 0.162491 | FG: 0.159496 | BG: 0.174472


Epoch 1/10 | Batch 50/1000 | Loss: 0.932684 | FG: 0.956748 | BG: 0.836427


...

Epoch 10/10 | Batch 960/1000 | Loss: 0.010964 | FG: 0.013080 | BG: 0.002499


Epoch 10/10 | Batch 970/1000 | Loss: 0.002785 | FG: 0.003137 | BG: 0.001378


Epoch 10/10 | Batch 980/1000 | Loss: 0.010100 | FG: 0.011532 | BG: 0.004370


Epoch 10/10 | Batch 990/1000 | Loss: 0.070355 | FG: 0.083127 | BG: 0.019263


Epoch 10/10 | Batch 1000/1000 | Loss: 0.011007 | FG: 0.012986 | BG: 0.003090
Epoch 10 completed | Loss: 0.011899 | FG: 0.013983 | BG: 0.003562
Saved: ddpm_v3_checkpoints/ddpm_epoch_010.pt
